In [1]:
pip install nba_api

Note: you may need to restart the kernel to use updated packages.


Standardization vs Normalization

In [2]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install numpy==1.23

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip list

Package                       Version
----------------------------- --------------------
absl-py                       1.3.0
alabaster                     0.7.12
anaconda-client               1.11.0
anaconda-navigator            2.3.2
anaconda-project              0.11.1
anyio                         3.5.0
appdirs                       1.4.4
argon2-cffi                   21.3.0
argon2-cffi-bindings          21.2.0
arrow                         1.2.2
astroid                       2.11.7
astropy                       5.1
astunparse                    1.6.3
atomicwrites                  1.4.0
attrs                         21.4.0
Automat                       20.2.0
autopep8                      1.6.0
Babel                         2.9.1
backcall                      0.2.0
backports.functools-lru-cache 1.6.4
backports.tempfile            1.0
backports.weakref             1.0.post1
bcrypt                        3.2.0
beautifulsoup4                4.11.1
binaryornot                   0.4.4
bi

https://towardsdatascience.com/how-different-metrics-correlate-with-winning-in-the-nba-over-30-years-57219d3d1c8
https://www.basketball-reference.com/leagues/NBA_2023.html

In [5]:
import numpy as np
import pandas as pd

In [6]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import teamyearbyyearstats
import datetime
from nba_api.stats.endpoints import LeagueDashTeamStats

team_names = ['Atlanta Hawks', 'Boston Celtics', 'Brooklyn Nets', 'Charlotte Hornets', 'Chicago Bulls', 'Cleveland Cavaliers', 
              'Dallas Mavericks', 'Denver Nuggets', 'Detroit Pistons', 'Golden State Warriors', 'Houston Rockets', 'Indiana Pacers', 
              'Los Angeles Clippers', 'Los Angeles Lakers', 'Memphis Grizzlies', 'Miami Heat', 'Milwaukee Bucks', 
              'Minnesota Timberwolves', 'New Orleans Pelicans', 'New York Knicks', 'Oklahoma City Thunder', 'Orlando Magic', 
              'Philadelphia 76ers', 'Phoenix Suns', 'Portland Trail Blazers', 'Sacramento Kings', 'San Antonio Spurs', 
              'Toronto Raptors', 'Utah Jazz', 'Washington Wizards']

example_home_team = "Boston Celtics"
example_away_team = "Miami Heat"

home_team_indicator = np.zeros(30)
away_team_indicator = np.zeros(30)

for i in range(len(team_names)):
    if (team_names[i] == example_home_team):
        home_team_indicator[i] = 1
    if (team_names[i] == example_away_team):
        away_team_indicator[i] = 1

current_year = "2022-23"

CurrentStats = LeagueDashTeamStats(season=current_year).get_data_frames()[0]
HomeCurrentStats = CurrentStats[CurrentStats['TEAM_NAME'] == example_home_team]
AwayCurrentStats = CurrentStats[CurrentStats['TEAM_NAME'] == example_away_team]

hometeam_stats_vector = HomeCurrentStats.iloc[0, 3:].values
awayteam_stats_vector = AwayCurrentStats.iloc[0, 3:].values
    
combined_vector = np.concatenate((home_team_indicator, away_team_indicator, hometeam_stats_vector, awayteam_stats_vector))
print(combined_vector)

x_test = combined_vector
y_test = [119, 111]
    

[0.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0
 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0
 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0
 0.0 0.0 0.0 0.0 0.0 0.0 57 25 0.695 3996.0 3460 7278 0.475 1315 3492
 0.377 1436 1769 0.812 796 2921 3717 2186 1095.0 521 430 318 1542 1568
 9671 535.0 1 2 2 2 1 13 15 14 2 2 6 24 28 4 20 3 7 7 7 27 6 4 7 25 4 1 44
 38 0.537 3961.0 3215 6991 0.46 980 2852 0.344 1567 1885 0.831 796 2533
 3329 1955 1106.0 655 243 309 1516 1643 8977 -26.0 1 11 11 11 18 30 26 26
 17 10 27 10 19 2 20 29 27 25 9 6 30 1 3 17 30 21]


In [7]:
import requests
from bs4 import BeautifulSoup
import time

years = range(2000, 2024)
all_time_stats = []

for year in years:
    # Define the URL of the webpage
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}.html"

    # Send an HTTP GET request to the URL
    response = requests.get(url)

    team_stats = []
    year_stats = []

    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    tbody = soup.find("table", id="advanced-team").find("tbody")

    for row in tbody.find_all("tr"):
        team_stats = []
        td_element = row.find("td", {"class": "left", "data-stat": "team"})
        team_name = td_element.text.strip()
        team_stats.append(team_name)

        td_element = row.find("td", {"class": "right", "data-stat": "off_rtg"})
        off_rtg = td_element.text
        team_stats.append(off_rtg)

        td_element = row.find("td", {"class": "right", "data-stat": "def_rtg"})
        def_rtg = td_element.text
        team_stats.append(def_rtg)

        td_element = row.find("td", {"class": "right", "data-stat": "net_rtg"})
        net_rtg = td_element.text
        team_stats.append(net_rtg)
        
        #td_element = row.find("td", {"class: right", "data-stat": ""})
        year_stats.append(team_stats)
    all_time_stats.append(year_stats)
    time.sleep(0.6)
print(all_time_stats)
print(len(all_time_stats))
print(len(all_time_stats[0]))

[[['Los Angeles Lakers*', '107.3', '98.2', '+9.1'], ['Portland Trail Blazers*', '107.9', '100.8', '+7.1'], ['San Antonio Spurs*', '105.0', '98.6', '+6.4'], ['Phoenix Suns*', '104.6', '99.0', '+5.6'], ['Utah Jazz*', '107.3', '102.3', '+5.0'], ['Indiana Pacers*', '108.5', '103.6', '+4.9'], ['Miami Heat*', '104.5', '101.0', '+3.5'], ['Sacramento Kings*', '105.0', '102.1', '+2.9'], ['Charlotte Hornets*', '104.3', '101.4', '+2.9'], ['Minnesota Timberwolves*', '106.1', '103.4', '+2.7'], ['New York Knicks*', '102.5', '100.9', '+1.6'], ['Detroit Pistons*', '107.3', '105.8', '+1.5'], ['Philadelphia 76ers*', '101.5', '100.0', '+1.5'], ['Seattle SuperSonics*', '105.6', '104.6', '+1.0'], ['Orlando Magic', '102.4', '101.7', '+0.7'], ['Milwaukee Bucks*', '108.2', '107.9', '+0.3'], ['Toronto Raptors*', '104.7', '104.9', '-0.2'], ['Dallas Mavericks', '106.6', '107.2', '-0.6'], ['Boston Celtics', '104.8', '105.6', '-0.8'], ['Houston Rockets', '104.8', '105.7', '-0.9'], ['New Jersey Nets', '105.1', '106

In [41]:
print(all_time_stats[0])
print(all_time_stats[0][0])
print(all_time_stats[0][0][0])
print(all_time_stats[0][1][0])
print(all_time_stats[0][0][1:])

[['Los Angeles Lakers*', '107.3', '98.2', '+9.1'], ['Portland Trail Blazers*', '107.9', '100.8', '+7.1'], ['San Antonio Spurs*', '105.0', '98.6', '+6.4'], ['Phoenix Suns*', '104.6', '99.0', '+5.6'], ['Utah Jazz*', '107.3', '102.3', '+5.0'], ['Indiana Pacers*', '108.5', '103.6', '+4.9'], ['Miami Heat*', '104.5', '101.0', '+3.5'], ['Sacramento Kings*', '105.0', '102.1', '+2.9'], ['Charlotte Hornets*', '104.3', '101.4', '+2.9'], ['Minnesota Timberwolves*', '106.1', '103.4', '+2.7'], ['New York Knicks*', '102.5', '100.9', '+1.6'], ['Detroit Pistons*', '107.3', '105.8', '+1.5'], ['Philadelphia 76ers*', '101.5', '100.0', '+1.5'], ['Seattle SuperSonics*', '105.6', '104.6', '+1.0'], ['Orlando Magic', '102.4', '101.7', '+0.7'], ['Milwaukee Bucks*', '108.2', '107.9', '+0.3'], ['Toronto Raptors*', '104.7', '104.9', '-0.2'], ['Dallas Mavericks', '106.6', '107.2', '-0.6'], ['Boston Celtics', '104.8', '105.6', '-0.8'], ['Houston Rockets', '104.8', '105.7', '-0.9'], ['New Jersey Nets', '105.1', '106.

In [42]:
game_finder = LeagueGameFinder(season_nullable="2019-20", league_id_nullable='00', season_type_nullable='Playoffs')
game_data = game_finder.get_data_frames()[0]
print(game_data)
playoff_data_scores = game_data[game_data['SEASON_ID'] == '42000']

    SEASON_ID     TEAM_ID TEAM_ABBREVIATION           TEAM_NAME     GAME_ID  \
0       42019  1610612748               MIA          Miami Heat  0041900406   
1       42019  1610612747               LAL  Los Angeles Lakers  0041900406   
2       42019  1610612748               MIA          Miami Heat  0041900405   
3       42019  1610612747               LAL  Los Angeles Lakers  0041900405   
4       42019  1610612748               MIA          Miami Heat  0041900404   
..        ...         ...               ...                 ...         ...   
161     42019  1610612738               BOS      Boston Celtics  0041900121   
162     42019  1610612746               LAC         LA Clippers  0041900151   
163     42019  1610612742               DAL    Dallas Mavericks  0041900151   
164     42019  1610612761               TOR     Toronto Raptors  0041900111   
165     42019  1610612755               PHI  Philadelphia 76ers  0041900121   

      GAME_DATE      MATCHUP WL  MIN  PTS  ...  FT_

In [43]:
#player injuries



In [8]:
import time
from nba_api.stats.endpoints import LeagueGameFinder
years = ["2000-01", "2001-02", "2002-03", "2003-04", "2004-05", "2005-06", "2006-07", "2007-08", "2008-09", "2009-10", "2010-11",
         "2011-12", "2012-13", "2013-14", "2014-15", "2015-16", "2016-17", "2017-18", "2018-19", "2019-20",
         "2020-21", "2021-22", "2022-23"]

# initialize the x train and y train sets
total_dataset = []

advanced_stat_yearcounter = 0

for year in years:
    time.sleep(0.6)
    # finding all playoff games for certain year
    game_finder = LeagueGameFinder(season_nullable=year, league_id_nullable='00', season_type_nullable='Playoffs')
    game_data = game_finder.get_data_frames()[0]
    
    #compiling home team names and away team names
    home_team_names = game_data.loc[game_data['MATCHUP'].str.contains('@'), 'TEAM_NAME'].tolist()
    away_team_names = game_data.loc[~game_data['MATCHUP'].str.contains('@'), 'TEAM_NAME'].tolist()
    
    #this to update the season id for each year
    extra = year[3]
    season_id = str(42000 + int(extra))
    
    #compiling home team scores and away team scores, should be equal to len(home_team_names)
    playoff_data_scores = game_data[game_data['SEASON_ID'] == season_id]
    home_team_scores = playoff_data_scores[playoff_data_scores['MATCHUP'].str.contains('vs.')]['PTS'].tolist()
    away_team_scores = playoff_data_scores[~playoff_data_scores['MATCHUP'].str.contains('vs.')]['PTS'].tolist()

    year_dataset = []
    advanced_stats = []
    
    for home_team, away_team, home_score, away_score in zip(home_team_names, away_team_names, home_team_scores, away_team_scores):
        #initialize 60 features for home team and away team
        time.sleep(0.6)
        home_team_indicator = np.zeros(30)
        away_team_indicator = np.zeros(30)
        
        for i, team in enumerate(team_names):
            if team == home_team:
                home_team_indicator[i] = 1
            if team == away_team:
                away_team_indicator[i] = 1
        
        home_team_stats = LeagueDashTeamStats(season=year).get_data_frames()[0]
        away_team_stats = LeagueDashTeamStats(season=year).get_data_frames()[0]
        
        home_team_stats = home_team_stats[home_team_stats['TEAM_NAME'] == home_team].iloc[0, 3:].values
        away_team_stats = away_team_stats[away_team_stats['TEAM_NAME'] == away_team].iloc[0, 3:].values
        
        home_team_advancedstats = []
        away_team_advancedstats = []

        for i in range(len(all_time_stats[0])):
            temp = all_time_stats[advanced_stat_yearcounter][i][0]
            if all_time_stats[advanced_stat_yearcounter][i][0] == home_team or temp[0:len(temp)-1] == home_team:
                home_team_advancedstats = all_time_stats[advanced_stat_yearcounter][i][1:]
            if all_time_stats[advanced_stat_yearcounter][i][0] == away_team or temp[0:len(temp)-1] == away_team:
                away_team_advancedstats = all_time_stats[advanced_stat_yearcounter][i][1:]

        combined_vector = np.concatenate((home_team_indicator, away_team_indicator, home_team_stats, home_team_advancedstats,
                                          away_team_stats, away_team_advancedstats))
        
        year_dataset.append((combined_vector, (home_score, away_score)))
    
    advanced_stat_yearcounter += 1
    print(year_dataset)
    total_dataset.extend(year_dataset)

[(array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56, 26, 0.683, 3981.0,
       3109, 6685, 0.465, 439, 1275, 0.344, 1594, 2333, 0.683, 1085, 2583,
       3668, 1888, 1184.0, 564, 490, 324, 1872, 7, 8251, 277.0, 1, 2, 2,
       2, 6, 4, 11, 3, 11, 9, 20, 10, 2, 29, 4, 7, 5, 9, 9, 25, 6, 1, 17,
       8, 3, 8, '107.3', '98.2', '+9.1', 56, 26, 0.683,
       3970.6133333333332, 2902, 6487, 0.447, 262, 803, 0.326, 1697, 2277,
       0.745, 1075, 2600, 3675, 1692, 1292.0, 690, 408, 458, 1673, 2,
       7763, 351.0, 1, 2, 2, 2, 12, 16, 19, 12, 28, 28, 26, 3, 4, 18, 5,
       5, 4, 21, 21, 5, 19, 20, 4, 27, 15, 4, '101.5', '100.0', '+1.5'],
      dtype=object), (96, 108)), (array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

[(array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 58, 24, 0.707, 3956.0,
       3150, 6840, 0.461, 510, 1439, 0.354, 1494, 2138, 0.699, 1022, 2607,
       3629, 1882, 1040.0, 625, 478, 354, 1823, 8, 8304, 584.0, 1, 2, 2,
       2, 20, 4, 8, 6, 6, 7, 13, 12, 6, 29, 14, 2, 3, 9, 2, 18, 8, 3, 22,
       17, 3, 2, '108.4', '104.8', '+3.6', 52, 30, 0.634, 3966.0, 3042,
       6816, 0.446, 403, 1194, 0.338, 1402, 1907, 0.735, 1039, 2515, 3554,
       1990, 1189.0, 716, 490, 439, 1734, 9, 7889, 341.0, 1, 5, 5, 5, 8,
       8, 9, 14, 17, 14, 22, 20, 17, 24, 12, 11, 8, 3, 15, 4, 6, 18, 12,
       10, 13, 5, '100.0', '105.5', '-5.5'], dtype=object), (107, 113)), (array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

[(array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 49, 33, 0.598, 3951.0,
       2906, 6585, 0.441, 346, 1041, 0.332, 1662, 2195, 0.757, 991, 2528,
       3519, 1887, 1212.0, 717, 374, 423, 1772, 5, 7820, 428.0, 1, 8, 8,
       8, 26, 17, 18, 14, 23, 22, 22, 5, 5, 15, 14, 12, 10, 7, 16, 5, 21,
       19, 14, 16, 14, 4, '104.0', '99.5', '+4.5', 60, 22, 0.732, 3966.0,
       2908, 6297, 0.462, 449, 1270, 0.354, 1591, 2194, 0.725, 939, 2556,
       3495, 1636, 1295.0, 629, 529, 422, 1672, 4, 7856, 444.0, 1, 1, 1,
       1, 14, 16, 27, 4, 11, 11, 11, 10, 6, 26, 21, 6, 11, 22, 23, 17, 1,
       18, 5, 22, 12, 3, '106.5', '99.7', '+6.8'], dtype=object), (88, 77)), (array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

[(array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56, 26, 0.683, 3971.0,
       3028, 6676, 0.454, 365, 1115, 0.327, 1631, 2352, 0.693, 1001, 2535,
       3536, 1948, 1132.0, 682, 379, 308, 1732, 5, 8052, 320.0, 1, 4, 4,
       4, 7, 4, 7, 4, 22, 20, 25, 5, 1, 28, 15, 7, 6, 4, 5, 10, 21, 2, 14,
       4, 3, 7, '107.2', '104.7', '+2.5', 54, 28, 0.659, 3956.0, 2747,
       6314, 0.435, 333, 968, 0.344, 1561, 2074, 0.753, 1014, 2492, 3506,
       1702, 1241.0, 659, 570, 410, 1664, 4, 7388, 479.0, 1, 6, 6, 6, 19,
       24, 26, 19, 24, 26, 15, 9, 9, 15, 12, 12, 11, 15, 18, 13, 1, 13, 4,
       9, 24, 2, '104.1', '99.9', '+4.2'], dtype=object), (100, 87)), (array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

[(array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 54, 28, 0.659, 3991.0,
       2851, 6421, 0.444, 363, 1053, 0.345, 1588, 2150, 0.739, 1054, 2507,
       3561, 1787, 1133.0, 576, 497, 367, 1638, 3, 7653, 317.0, 1, 5, 5,
       5, 3, 23, 23, 17, 22, 22, 23, 16, 13, 23, 6, 10, 4, 12, 11, 20, 3,
       8, 2, 13, 24, 6, '102.0', '95.4', '+6.6', 59, 23, 0.72, 3961.0,
       2923, 6450, 0.453, 507, 1395, 0.363, 1535, 2120, 0.724, 987, 2489,
       3476, 1770, 1126.0, 613, 543, 421, 1717, 4, 7888, 640.0, 1, 2, 2,
       2, 17, 19, 21, 10, 12, 13, 8, 23, 16, 26, 15, 11, 12, 14, 8, 17, 2,
       23, 5, 8, 18, 1, '102.2', '94.1', '+8.1'], dtype=object), (81, 74)), (array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1

...

[(array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 58, 24, 0.707, 3956.0,
       2999, 6328, 0.474, 595, 1561, 0.381, 1486, 1980, 0.751, 761, 2577,
       3338, 1814, 1137.0, 587, 417, 337, 1588, 1739, 8079, 691.0, 1, 3,
       3, 3, 22, 12, 27, 3, 6, 7, 3, 24, 24, 17, 28, 4, 16, 11, 4, 15, 10,
       8, 1, 24, 14, 1, '107.3', '99.6', '+7.7', 50, 32, 0.61, 3976.0,
       2978, 6658, 0.447, 494, 1404, 0.352, 1484, 2133, 0.696, 1039, 2529,
       3568, 1708, 1177.0, 625, 353, 349, 1781, 1804, 7934, 314.0, 1, 7,
       7, 7, 8, 15, 9, 24, 15, 14, 18, 25, 15, 29, 1, 6, 2, 15, 7, 8, 20,
       10, 11, 16, 19, 7, '107.8', '105.4', '+2.4'], dtype=object), (82, 83)), (array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

[(array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 57, 25, 0.695, 3956.0,
       3248, 6818, 0.476, 662, 1751, 0.378, 1746, 2270, 0.769, 898, 2722,
       3620, 2003, 1156.0, 654, 438, 368, 1691, 1850, 8904, 595.0, 1, 3,
       3, 3, 18, 5, 6, 3, 5, 6, 6, 3, 5, 10, 19, 1, 4, 4, 11, 6, 5, 12,
       15, 5, 4, 3, '108.6', '108.6', '0.0', 66, 16, 0.805, 3951.0, 2986,
       6286, 0.475, 596, 1564, 0.381, 1677, 2176, 0.771, 830, 2615, 3445,
       1833, 1246.0, 696, 379, 389, 1864, 1818, 8245, 841.0, 1, 1, 1, 1,
       22, 20, 30, 4, 8, 12, 5, 8, 9, 8, 23, 9, 12, 8, 27, 5, 18, 16, 26,
       8, 11, 1, '103.2', '106.9', '-3.7'], dtype=object), (131, 92)), (array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.

[(array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 65, 17, 0.793, 3956.0,
       3307, 6981, 0.474, 547, 1516, 0.361, 1607, 2087, 0.77, 1015, 2587,
       3602, 1908, 1103.0, 718, 420, 392, 1698, 1816, 8768, 628.0, 1, 2,
       2, 2, 21, 2, 4, 4, 17, 15, 19, 11, 8, 14, 3, 6, 1, 2, 11, 2, 10,
       14, 17, 8, 3, 2, '113.0', '105.5', '+7.5', 59, 23, 0.72, 3946.0,
       2929, 6416, 0.457, 817, 2147, 0.381, 1611, 2253, 0.715, 819, 2728,
       3547, 1593, 1142.0, 570, 439, 308, 1664, 1840, 8286, 549.0, 1, 4,
       4, 4, 27, 26, 26, 17, 2, 2, 7, 10, 4, 30, 27, 1, 3, 29, 12, 22, 6,
       2, 10, 6, 10, 4, '111.3', '105.5', '+5.8'], dtype=object), (86, 99)), (array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

[(array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
       0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 50, 32, 0.61, 3956.0, 3039,
       6294, 0.483, 499, 1433, 0.348, 1559, 2090, 0.746, 716, 2449, 3165,
       1930, 1219.0, 701, 402, 386, 1816, 1777, 8136, 300.0, 1, 9, 9, 9,
       17, 21, 30, 4, 16, 16, 17, 15, 10, 21, 30, 25, 29, 2, 21, 2, 15,
       15, 22, 8, 19, 9, '110.5', '102.3', '+8.2', 57, 25, 0.695, 3966.0,
       3144, 6875, 0.457, 532, 1562, 0.341, 1519, 1985, 0.765, 973, 2662,
       3635, 1730, 1096.0, 612, 400, 363, 1592, 1737, 8339, 387.0, 1, 3,
       3, 3, 12, 9, 7, 18, 13, 10, 24, 17, 17, 12, 4, 4, 2, 15, 6, 11, 16,
       9, 4, 12, 12, 6, '112.8', '104.7', '+8.1'], dtype=object), (83, 79)), (array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]


In [ ]:
print(total_dataset[-1])
print(home_team)
print(away_team)
print(year)

In [34]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

from tensorflow.keras.layers.experimental.preprocessing import Normalization
from tensorflow.keras import regularizers

total_dataset = np.array(total_dataset)
x_train = np.array([data[0] for data in total_dataset if np.shape(data[0]) == (168,)])
y_train = np.array([data[1] for data in total_dataset if np.shape(data[1]) == (2,)])


#print([np.shape(data[0]) for data in total_dataset[:5]])
#print([np.shape(data[1]) for data in total_dataset[:5]])

#needed to convert numpy array to tensor
x_train = x_train.astype(np.float64)
y_train = y_train.astype(np.float64)

# Initialize the Normalization layer and adapt it to the training data
normalizer = Normalization(axis=-1)
normalizer.adapt(x_train)

#fitting on the training set
#min_max_scaler.fit(x_train)

#x_train_normalized = min_max_scaler.transform(x_train)
#x_test_normalized = min_max_scaler.transform(x_test)

# Create the model
model = Sequential()
model.add(normalizer)
# Add layers
model.add(layers.Dense(128, activation='relu', input_shape = (168,), kernel_regularizer=regularizers.l2(0.01)))
layers.Dropout(0.5),
model.add(layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01)))
layers.Dropout(0.5),
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dense(16, activation='relu'))
layers.Dropout(0.5),
model.add(Dense(2))

# Print the model summary
model.summary()

model.compile(optimizer='adam', loss='mean_squared_error', metrics =['mse'])

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model.fit(x_train, y_train, epochs=50, batch_size=32, validation_split=0.2, callbacks=[early_stopping])



Model: "sequential_13"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 normalization_11 (Normaliza  (None, 168)              337       
 tion)                                                           
                                                                 
 dense_59 (Dense)            (None, 128)               21632     
                                                                 
 dense_60 (Dense)            (None, 64)                8256      
                                                                 
 dense_61 (Dense)            (None, 32)                2080      
                                                                 
 dense_62 (Dense)            (None, 16)                528       
                                                                 
 dense_63 (Dense)            (None, 2)                 34        
                                                     

In [15]:
from nba_api.stats.endpoints import TeamDashboardByYearOverYear
from nba_api.stats.endpoints import LeagueLeaders

teamstats = TeamDashboardByYearOverYear(season=current_year, team_id = 2).get_data_frames()[0]
teamanalyticsstats = LeagueDashTeamStats(season=current_year).get_data_frames()[0]
leaders = LeagueLeaders(season=current_year).get_data_frames()[0]

print(teamstats)
print(teamanalyticsstats)
print(leaders)


Empty DataFrame
Columns: [GROUP_SET, GROUP_VALUE, GP, W, L, W_PCT, MIN, FGM, FGA, FG_PCT, FG3M, FG3A, FG3_PCT, FTM, FTA, FT_PCT, OREB, DREB, REB, AST, TOV, STL, BLK, BLKA, PF, PFD, PTS, PLUS_MINUS, GP_RANK, W_RANK, L_RANK, W_PCT_RANK, MIN_RANK, FGM_RANK, FGA_RANK, FG_PCT_RANK, FG3M_RANK, FG3A_RANK, FG3_PCT_RANK, FTM_RANK, FTA_RANK, FT_PCT_RANK, OREB_RANK, DREB_RANK, REB_RANK, AST_RANK, TOV_RANK, STL_RANK, BLK_RANK, BLKA_RANK, PF_RANK, PFD_RANK, PTS_RANK, PLUS_MINUS_RANK]
Index: []

[0 rows x 54 columns]
       TEAM_ID               TEAM_NAME  GP   W   L  W_PCT     MIN   FGM   FGA  \
0   1610612737           Atlanta Hawks  82  41  41  0.500  3971.0  3658  7574   
1   1610612738          Boston Celtics  82  57  25  0.695  3996.0  3460  7278   
2   1610612751           Brooklyn Nets  82  45  37  0.549  3946.0  3399  6978   
3   1610612766       Charlotte Hornets  82  27  55  0.329  3966.0  3385  7413   
4   1610612741           Chicago Bulls  82  40  42  0.488  3981.0  3488  7116   
5   1